# Enhancing PV Grid Management Through AC-Power Prediction

<img src="assets/Project Header.jpg" alt="Solar photovoltaic array" width="900">

**Can weather and time predict a plant's AC output accurately enough to support grid operations?**

## Problem Statement

Solar output is intermittent. A grid operator scheduling generation for the next block needs to know how much
power a photovoltaic plant will deliver, not how much it could deliver under perfect conditions. The physical
inputs are observable — irradiance, air temperature, and the plant itself — but the relationship between them and
AC power is non-linear, sensitive to panel temperature, and specific to each site.

This notebook studies two adjacent PV plants in India. Each has 22 inverters reporting every 15 minutes for 34 days.
The task is to predict **plant-level AC power in kW** from inputs a dispatch team could actually have before the
interval starts: forecast irradiation, forecast ambient temperature, the hour of day, and the plant identity.

The analysis is deliberately conservative. It holds out the final seven days chronologically rather than sampling
rows at random, reports the untuned screening results for every candidate, and ships only the two models that earn
their place on cross-validated error.

## Goals and Objectives

**Goal:** produce a defensible AC-power model plus the artifacts a downstream application can load without
re-implementing any preprocessing.

**Objectives**

1. Merge inverter-level generation with plant-level weather into one timestamp frame, with the aggregation rule
   stated explicitly rather than implied.
2. Restrict modelling to daylight, where output is non-trivial, and say plainly that night output is zero.
3. Screen eleven regression algorithms on a chronological split and report cross-validated and held-out error for
   each — not just for the winner.
4. Tune only the two best candidates, guard against tuning making them worse, and persist each with its feature
   order and metrics.

## Metrics

| Metric | Unit | Why it is used |
| --- | --- | --- |
| **MAE** | kW | Average absolute miss; easy to explain to an operator. |
| **RMSE** | kW | Penalises large misses, which matter most for reserve scheduling. |
| **R²** | — | Share of variance explained; comparable across models, but inflated by any trivial structure left in the data. |
| **MAPE** | % | Reported for completeness only. Output approaches zero at dawn and dusk, so percentage error explodes. |

## Data and Units

Source: Kaggle *Solar Power Generation Data* (two Indian plants, 15-minute readings, 2020-05-15 to 2020-06-17).
Generation files are per-inverter; weather files are per-plant.

- `AC_POWER` in this notebook is the **sum across the plant's 22 inverters** at each timestamp, in kW.
- `IRRADIATION` is in kW/m², `AMBIENT_TEMPERATURE` and `MODULE_TEMPERATURE` in °C.
- "Daylight" means `IRRADIATION > 0.05` kW/m². That threshold removes dawn/dusk noise and the night zeros.

# **STAGE 1 : EXPLORATORY DATA ANALYSIS (EDA)**

## Load the Raw Files

Generation timestamps do not even agree between the two plants: Plant 1 stores `15-05-2020 00:00` (day first), Plant 2
stores `2020-05-15 00:00:00` (ISO), and the weather files are ISO. Each file is parsed with an explicit format chosen
from its first value, so day and month can never be silently swapped. `format="mixed"` is not used: with
`dayfirst=True` it mis-parses the ISO file, and without it the day-first file is ambiguous for days 1-12.

The two file types have different granularity: generation is **per inverter** (22 rows per timestamp), weather is
**per plant** (one row per timestamp). That difference decides the merge strategy in the next cell.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from utils import (
    check_data_information,
    describe_numerical_combined,
    plot_correlation_heatmap,
    plot_dynamic_boxplots_violinplots,
    plot_dynamic_hisplots_kdeplots,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_DIR = Path("data")
MODELS_DIR = Path("models")
TARGET = "AC_POWER"
RANDOM_STATE = 42
DAYLIGHT_THRESHOLD = 0.05
PLANT_SOURCE_IDS = {4135001: 1, 4135002: 2}
PLANT_NAMES = {1: "Plant 1", 2: "Plant 2"}

In [ ]:
def parse_generation_dates(values: pd.Series) -> pd.Series:
    """Parse the generation format, which differs between the two plants."""
    sample = str(values.iloc[0])
    if sample[:4].isdigit() and sample[4:5] == "-":
        return pd.to_datetime(values, format="%Y-%m-%d %H:%M:%S")
    return pd.to_datetime(values, format="%d-%m-%Y %H:%M")


generation_raw = {}
sensor_raw = {}

for source_id, plant_id in PLANT_SOURCE_IDS.items():
    generation = pd.read_csv(DATA_DIR / f"Plant_{plant_id}_Generation_Data.csv")
    generation["DATE_TIME"] = parse_generation_dates(generation["DATE_TIME"])
    sensor = pd.read_csv(
        DATA_DIR / f"Plant_{plant_id}_Weather_Sensor_Data.csv",
        parse_dates=["DATE_TIME"],
    )
    generation_raw[plant_id] = generation
    sensor_raw[plant_id] = sensor
    print(
        f"{PLANT_NAMES[plant_id]}: generation {generation.shape}, "
        f"sensor {sensor.shape}, inverters {generation.SOURCE_KEY.nunique()}"
    )

display(generation_raw[1].head(3))
display(sensor_raw[1].head(3))

## One Plant's DC Channel Cannot Be Trusted

Before anything is aggregated, the DC and AC channels are compared on the same inverter rows. Plant 2 behaves as
a PV plant should: DC input is slightly above AC output. Plant 1's `DC_POWER` is roughly ten times its `AC_POWER`
on every row, which is physically impossible for the same inverter.

The cause cannot be recovered from the files available here. The consequence is actionable: **`DC_POWER` is
excluded from every later step**, and no DC-side efficiency claim is made anywhere in this notebook.

In [ ]:
ratio_rows = []
for plant_id, generation in generation_raw.items():
    totals = generation.groupby("DATE_TIME")[["DC_POWER", "AC_POWER"]].sum()
    ratio = (totals.DC_POWER / totals.AC_POWER.replace(0, np.nan)).dropna()
    ratio_rows.append(
        {
            "Plant": PLANT_NAMES[plant_id],
            "median DC/AC": ratio.median(),
            "p95 DC/AC": ratio.quantile(0.95),
        }
    )

dc_ratio = pd.DataFrame(ratio_rows).round(2)
display(dc_ratio)

assert 9 < dc_ratio.loc[0, "median DC/AC"] < 11, "Plant 1 DC anomaly not detected"
assert 0.9 < dc_ratio.loc[1, "median DC/AC"] < 1.1, "Plant 2 DC/AC looks wrong"

## Build the Analysis Frame

The merge rule follows from the granularity:

1. **Sum `AC_POWER` across inverters** per timestamp. Summing `DC_POWER` would import the Plant 1 anomaly.
2. **Merge weather 1:1 on timestamp.** The weather file has one row per timestamp, so summing it — as the earlier
   version of this notebook did — multiplies temperature by the inverter count.
3. Recode the source plant IDs to `1`/`2` immediately, so every later cell, chart, and artifact uses the same
   canonical label.

The inner merge also drops weather timestamps that have no generation record; that is intentional.

In [ ]:
frames = []
for plant_id, generation in generation_raw.items():
    ac_only = generation.groupby("DATE_TIME", as_index=False)[["AC_POWER"]].sum()
    sensor = sensor_raw[plant_id].drop(columns=["PLANT_ID", "SOURCE_KEY"])
    merged = ac_only.merge(sensor, on="DATE_TIME", how="inner")
    merged["PLANT_ID"] = plant_id
    frames.append(merged)

df = pd.concat(frames, ignore_index=True)
df["PLANT_NAME"] = df["PLANT_ID"].map(PLANT_NAMES)
df["DATE"] = df["DATE_TIME"].dt.date
df["HOUR"] = df["DATE_TIME"].dt.hour
df = df.sort_values(["DATE_TIME", "PLANT_ID"]).reset_index(drop=True)

assert len(df) == 6416, f"unexpected row count: {len(df)}"
assert df.isna().sum().sum() == 0, "unexpected nulls after merge"
assert df.duplicated(["DATE_TIME", "PLANT_ID"]).sum() == 0

print(f"rows: {len(df):,}")
print(f"per plant: {df.PLANT_ID.value_counts().sort_index().to_dict()}")
print(f"window: {df.DATE_TIME.min()} -> {df.DATE_TIME.max()}")
print(f"days: {df.DATE_TIME.dt.date.nunique()}")

## Data Understanding

In [ ]:
display(check_data_information(df, df.columns.tolist()))

In [ ]:
audit = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "unique": df.nunique(),
    }
)
display(audit)

print("duplicate (timestamp, plant) rows:", int(df.duplicated(["DATE_TIME", "PLANT_ID"]).sum()))

## Descriptive Statistics

In [ ]:
NUMERIC_COLS = ["AC_POWER", "IRRADIATION", "AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "HOUR"]
display(describe_numerical_combined(df, NUMERIC_COLS))

## Univariate Analysis

### Numerical Columns

The target is strongly zero-inflated because half of the 15-minute intervals are night. That is not a modelling
problem to fix with a transform; it is a reason to restrict the model population to daylight in Stage 2.

In [ ]:
plot_dynamic_hisplots_kdeplots(df=df, col_series=NUMERIC_COLS, ncols=3, figsize=(16, 10))

In [ ]:
plot_dynamic_boxplots_violinplots(
    df=df,
    col_series=NUMERIC_COLS,
    plot_type="boxplot",
    ncols=3,
    orientation="h",
    figsize=(16, 10),
)

### Correlation

Spearman correlation is used throughout: the relationships are monotone but not linear, and rank correlation is
not distorted by the night pile-up at zero.

In [ ]:
plot_correlation_heatmap(df, col_series=NUMERIC_COLS, corr_method="spearman", figsize=(10, 8))

## Bivariate and Multivariate Analysis

Correlation over the full 24-hour frame flatters irradiation because both it and the target are zero at night.
The table below recomputes the relationship **within daylight only**, where the model will actually operate.

In [ ]:
DAYLIGHT = df[df.IRRADIATION > DAYLIGHT_THRESHOLD].copy()
CORR_COLS = ["AC_POWER", "IRRADIATION", "AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE"]

corr_by_plant = pd.concat(
    {
        name: group[CORR_COLS].corr()["AC_POWER"]
        for name, group in DAYLIGHT.groupby("PLANT_NAME")
    },
    axis=1,
).drop(index="AC_POWER")

display(corr_by_plant.round(3))
print(f"daylight rows: {len(DAYLIGHT):,}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, group in DAYLIGHT.groupby("PLANT_NAME"):
    ax.scatter(group.IRRADIATION, group.AC_POWER, s=8, alpha=0.30, label=name)

ax.set_xlabel("Irradiation (kW/m²)")
ax.set_ylabel("AC power (kW)")
ax.set_title("Daylight irradiation vs AC power")
ax.legend(title="")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

hourly = DAYLIGHT.groupby(["HOUR", "PLANT_NAME"], as_index=False)["AC_POWER"].mean()
sns.lineplot(
    data=hourly, x="HOUR", y="AC_POWER", hue="PLANT_NAME", marker="o", ax=axes[0]
)
axes[0].set_title("Mean daylight AC power by hour")
axes[0].set_xlabel("Hour of day")
axes[0].set_ylabel("kW")

daily = DAYLIGHT.groupby(["DATE", "PLANT_NAME"], as_index=False)["AC_POWER"].mean()
daily["DATE"] = pd.to_datetime(daily["DATE"])
sns.lineplot(data=daily, x="DATE", y="AC_POWER", hue="PLANT_NAME", ax=axes[1])
axes[1].set_title("Mean daylight AC power by day")
axes[1].set_xlabel("")
axes[1].set_ylabel("kW")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Zero-Output Rows During Daylight

In [ ]:
outages = DAYLIGHT[DAYLIGHT.AC_POWER == 0]
display(outages[["DATE_TIME", "PLANT_NAME", "IRRADIATION", "AMBIENT_TEMPERATURE"]])
print("zero-output daylight rows:", len(outages))
print(outages.PLANT_NAME.value_counts().to_dict())

In [ ]:
summary = (
    DAYLIGHT.groupby("PLANT_NAME")
    .agg(
        rows=("AC_POWER", "size"),
        peak_kw=("AC_POWER", "max"),
        mean_kw=("AC_POWER", "mean"),
        max_irradiation=("IRRADIATION", "max"),
        max_module_temp=("MODULE_TEMPERATURE", "max"),
    )
    .round(2)
)
display(summary)

## What the EDA Established

- **Irradiation is close to a sufficient statistic for Plant 1, but not for Plant 2.** Within daylight, the rank
  correlation with AC power is 0.990 for Plant 1 and 0.751 for Plant 2. The gap is the first hint that a single
  pooled model must know which plant it is scoring.
- **The plants differ in scale.** Peak AC power is 29,150 kW at Plant 1 and 25,980 kW at Plant 2. Pooling without
  a plant identifier would average two different capacities.
- **Nine daylight rows at Plant 2 report zero output under strong irradiation.** These are outages, not weather
  effects. They are kept, counted, and treated as irreducible error.
- **Plant 1's DC channel is unusable**, so no DC-side or efficiency analysis appears later.
- **The window is short.** Thirty-four days in one season cannot support a claim about annual behaviour.
- **Module temperature tracks irradiation closely** (0.906 and 0.674 within daylight) but is a panel-mounted
  sensor reading, not a forecast input. Stage 3 measures what excluding it costs.

The modelling frame is now defined. Stage 2 narrows it to daylight, splits it chronologically, and checks the
outlier and scaling decisions before any model is fitted.

# **STAGE 2 : DATA PREPROCESSING & FEATURE ENGINEERING**

## Daylight Population

Roughly half of the 15-minute intervals are night, where irradiation and AC power are both zero. Keeping them
would inflate R² with a trivial on/off pattern and dominate every residual plot with rows the model cannot get
wrong. The modelling population is therefore **daylight only**: `IRRADIATION > 0.05` kW/m².

The application this notebook feeds treats any interval at or below that threshold as **0 kW by definition**; the
model is never asked to score night.

In [ ]:
df_day = df[df.IRRADIATION > DAYLIGHT_THRESHOLD].reset_index(drop=True)

assert len(df_day) == 3039, f"unexpected daylight rows: {len(df_day)}"
assert df_day.PLANT_ID.value_counts().to_dict() == {2: 1526, 1: 1513}

print(f"daylight rows: {len(df_day):,}")
print(f"per plant: {df_day.PLANT_ID.value_counts().sort_index().to_dict()}")
print(f"window: {df_day.DATE_TIME.min()} -> {df_day.DATE_TIME.max()}")

## Chronological Train-Test Split

The final seven days are held out. A random row split would place the same day — and near-duplicate 15-minute
readings a quarter-hour apart — on both sides of the split, so the test score would measure interpolation, not
forecasting.

The holdout answers the question an operator asks: **how well does this score tomorrow?** It is reported once and
never used to choose a model; selection happens on cross-validation inside the training window.

In [ ]:
cutoff = df_day.DATE_TIME.max().normalize() - pd.Timedelta(days=7)
train_df = df_day[df_day.DATE_TIME < cutoff].reset_index(drop=True)
test_df = df_day[df_day.DATE_TIME >= cutoff].reset_index(drop=True)

assert cutoff == pd.Timestamp("2020-06-10 00:00")
assert len(train_df) == 2322, f"unexpected train rows: {len(train_df)}"
assert len(test_df) == 717, f"unexpected test rows: {len(test_df)}"
assert test_df.DATE_TIME.min() == pd.Timestamp("2020-06-10 06:30")

print(f"cutoff: {cutoff}")
print(f"train: {len(train_df):,} rows | {train_df.DATE_TIME.min()} -> {train_df.DATE_TIME.max()}")
print(f"test:  {len(test_df):,} rows | {test_df.DATE_TIME.min()} -> {test_df.DATE_TIME.max()}")
print(f"test per plant: {test_df.PLANT_ID.value_counts().sort_index().to_dict()}")

## Candidate Inputs

The candidate list is deliberately narrow: everything on it can be known before the interval starts.

- `IRRADIATION` — forecast irradiance, the dominant physical driver.
- `AMBIENT_TEMPERATURE` — forecast air temperature; also a proxy for panel heating.
- `HOUR` — time of day; captures sun angle and the thermal lag that irradiance alone misses. It is a candidate
  here, not an assumption; Stage 3 decides whether it earns a place.
- `PLANT_ID` — the pooled model's only way to know which plant it is scoring. The plants differ in capacity by
  about 12%, so pooling without it would be wrong.

`MODULE_TEMPERATURE` is **not** on the list. It is a panel-mounted sensor reading, unavailable to a dispatch team
before the interval. Stage 3 measures what excluding it costs instead of asserting that it is free.

In [ ]:
FEATURE_CANDIDATES = ["IRRADIATION", "AMBIENT_TEMPERATURE", "HOUR", "PLANT_ID"]

X_train = train_df[FEATURE_CANDIDATES].copy()
X_test = test_df[FEATURE_CANDIDATES].copy()
y_train = train_df[TARGET].copy()
y_test = test_df[TARGET].copy()

print("train matrix:", X_train.shape)
print("test matrix: ", X_test.shape)
display(X_train.head())

## Outlier Check on the Training Split

IQR flags are inspected, not applied. On this training window it flags **nothing**: once the night rows are
gone, the physical ranges are well behaved. If it had flagged clear-sky midday peaks, the decision would still be
to keep them — a peak is exactly the regime the model exists to predict, and clipping it would improve the metrics
while making the model worse where it matters. The test split is never touched by any cleaning decision.

In [ ]:
from utils import filter_outliers

outlier_summary = filter_outliers(
    train_df,
    columns=["IRRADIATION", "AMBIENT_TEMPERATURE"],
    method="iqr",
    threshold=1.5,
    detect_only=True,
)
display(outlier_summary)

## Feature Scaling

Scale-sensitive candidates — linear, distance-based, and neural models — need bounded inputs. Min-max is fitted on
the training split and the **same bounds** are applied to the test split.

Artifacts must accept raw values, so whenever a scale-sensitive model is shipped, the identical transform is
embedded inside its `Pipeline` rather than kept as a separate object. Tree ensembles are fitted on raw values and
need no scaler at all.

In [ ]:
from utils import feature_scaling

X_train_scaled, scaler = feature_scaling(
    X_train, columns=FEATURE_CANDIDATES, method="minmax"
)
X_test_scaled, _ = feature_scaling(
    X_test, columns=FEATURE_CANDIDATES, method="minmax", scaler=scaler
)

display(X_train_scaled[FEATURE_CANDIDATES].describe().transpose()[["min", "max"]].round(3))

**Stage 2 decisions, recorded:**

| Decision | Choice |
| --- | --- |
| Population | Daylight only, 3,039 rows |
| Split | Chronological at 2020-06-10, 2,322 train / 717 test |
| Outliers | Detected and reported; not removed |
| Scaling | Fitted on train, applied to test; embedded in pipelines for scale-sensitive artifacts |

# **STAGE 3 : FEATURE SELECTION**

## Correlation Structure

Two questions are separate and need separate tools:

1. **Do the candidates carry the target signal?** Rank correlation and model evidence answer this.
2. **Do the candidates duplicate each other?** Variance inflation answers this; a correlation table alone does not.

In [ ]:
from utils import calculate_correlation_tabular

feature_feature_df, feature_target_df = calculate_correlation_tabular(
    train_df, target_col=TARGET, method="spearman", corr_type="both"
)

display(feature_target_df)
display(feature_feature_df)

## Multicollinearity Check (VIF)

VIF above 5 is moderate redundancy and above 10 is severe. The table includes the target column because the helper
needs it in the frame; the target is removed before any variance is computed.

In [ ]:
from utils import calculate_vif

vif_data = calculate_vif(
    train_df[FEATURE_CANDIDATES + [TARGET]],
    target_col=TARGET,
    high_threshold=10,
    moderate_threshold=5,
)
display(vif_data)

## What Excluding Module Temperature Costs

`MODULE_TEMPERATURE` is the one variable that clearly predicts output but cannot be known before dispatch. Rather
than assert that excluding it is free, the same default Random Forest is fitted three ways on the training split:
the forecastable set, that set without `HOUR`, and that set plus module temperature.

All three are evaluated with the same 3-fold, 1-repeat cross-validation and the same chronological holdout. The
delta column is what each choice buys or costs in CV R².

In [ ]:
from utils import eval_regression, get_model_pipeline


def quick_cv(features: list[str]) -> dict:
    """Fit a default Random Forest on `features` and evaluate it."""
    estimator = get_model_pipeline("randomforest").fit(train_df[features], y_train)
    return eval_regression(
        model=estimator,
        X_test=test_df[features],
        y_test=y_test,
        X_train=train_df[features],
        y_train=y_train,
        n_splits=3,
        n_repeats=1,
        plot_predicted_vs_actual=False,
        plot_residuals=False,
        plot_residuals_dist=False,
        display=False,
        show_plots=False,
    )


def summarize(name: str, metrics: dict) -> dict:
    """Reduce an evaluation result to the three numbers this stage needs."""
    return {
        "feature_set": name,
        "cv_r2": metrics["cv"]["r2"]["test_mean"],
        "cv_rmse": metrics["cv"]["rmse"]["test_mean"],
        "holdout_r2": metrics["r2"]["test"],
    }


forecast_only = quick_cv(FEATURE_CANDIDATES)
without_hour = quick_cv([f for f in FEATURE_CANDIDATES if f != "HOUR"])
with_module = quick_cv(FEATURE_CANDIDATES + ["MODULE_TEMPERATURE"])

feature_evidence = pd.DataFrame(
    [
        summarize("forecastable (base)", forecast_only),
        summarize("forecastable, no HOUR", without_hour),
        summarize("forecastable + module temp", with_module),
    ]
)
feature_evidence["delta_cv_r2"] = feature_evidence["cv_r2"] - feature_evidence.loc[0, "cv_r2"]

display(feature_evidence.round(4))

## Final Feature Set

The decision rule is explicit:

- **Keep a forecastable candidate** when it adds at least 0.005 CV R². `HOUR` adds roughly 0.011, so it stays.
- **Exclude a feature that cannot be known before dispatch**, regardless of its gain. Module temperature adds about
  0.0065 CV R² — real, but small, and it would make the model unusable for the task it exists to do. The cost is
  reported above instead of hidden.
- **Drop any candidate with severe VIF.** None of the four crosses 5, so no linear redundancy removal is needed.

The result is four inputs, all available to a dispatch team before the interval starts.

In [ ]:
FEATURE_COLUMNS = ["IRRADIATION", "AMBIENT_TEMPERATURE", "HOUR", "PLANT_ID"]

assert set(FEATURE_COLUMNS) <= set(FEATURE_CANDIDATES)
assert "MODULE_TEMPERATURE" not in FEATURE_COLUMNS
assert len(FEATURE_COLUMNS) == 4

# Rebuild the modelling matrices on the final feature list and order.
X_train = train_df[FEATURE_COLUMNS].copy()
X_test = test_df[FEATURE_COLUMNS].copy()

print("final features:", FEATURE_COLUMNS)
print("train:", X_train.shape, "| test:", X_test.shape)

# **STAGE 4 : MODELLING & EVALUATION**

## Candidate Models

Nine regressors are screened, all from the vendored toolkit's `get_model_pipeline` registry:

`LinearRegression`, `Ridge`, `Lasso`, `ElasticNet`, `KNN`, `DecisionTree`, `RandomForest`,
`GradientBoosting`, `XGBoost`.

Three common candidates are deliberately absent. `AdaBoost` and `MLP` have no pipeline or hyperparameter entry in
`utils`, so including them would mean hand-building two special cases with no tuning path. `SVR` is excluded on
runtime grounds: its training cost grows roughly quadratically with sample count.

Scale-sensitive models get a `MinMaxScaler` embedded in their own pipeline. That keeps every estimator
interchangeable at the call site and means any scale-sensitive winner can be persisted as a self-contained object
that accepts raw values.

In [ ]:
import time

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

from utils import get_model_pipeline

SCALE_SENSITIVE = {"linearregression", "ridge", "lasso", "elasticnet", "knn"}

MODEL_REGISTRY_KEYS = {
    "LinearRegression": "linearregression",
    "Ridge": "ridge",
    "Lasso": "lasso",
    "ElasticNet": "elasticnet",
    "KNN": "knn",
    "DecisionTree": "decisiontree",
    "RandomForest": "randomforest",
    "GradientBoosting": "gb",
    "XGBoost": "xgboost",
}
MODEL_NAMES = list(MODEL_REGISTRY_KEYS)


def build_estimator(name: str) -> Pipeline:
    """Return an unfitted pipeline; scale-sensitive models get an embedded scaler."""
    key = MODEL_REGISTRY_KEYS[name]
    pipeline = get_model_pipeline(key)
    if key not in SCALE_SENSITIVE:
        return pipeline
    step, estimator = pipeline.steps[0]
    return Pipeline([("scaler", MinMaxScaler()), (step, estimator)])


print("candidates:", len(MODEL_NAMES))
print({name: [step for step, _ in build_estimator(name).steps] for name in MODEL_NAMES})

## Screening Protocol

Every candidate is fitted on the training window and evaluated twice:

- **Cross-validation (3-fold, 1 repeat)** inside the training window — this is what ranks the models.
- **Chronological holdout** — reported for every candidate, but never used to choose. Using the holdout for
  selection would leak the test set into the decision and make the final score meaningless.

`eval_regression` evaluates an **already-fitted** estimator; it predicts on the test split before running
cross-validation. Each model is therefore fitted explicitly before the call.

In [ ]:
from utils import compare_cv_metrics, eval_regression

screening_metrics = {}
screening_seconds = {}

for name in MODEL_NAMES:
    started = time.perf_counter()
    estimator = build_estimator(name).fit(X_train, y_train)
    screening_metrics[name] = eval_regression(
        model=estimator,
        X_test=X_test,
        y_test=y_test,
        X_train=X_train,
        y_train=y_train,
        n_splits=3,
        n_repeats=1,
        plot_predicted_vs_actual=False,
        plot_residuals=False,
        plot_residuals_dist=False,
        display=False,
        show_plots=False,
    )
    screening_seconds[name] = time.perf_counter() - started
    print(f"{name:18s} {screening_seconds[name]:6.1f}s")

## Screening Comparison

In [ ]:
screening_table = compare_cv_metrics(screening_metrics, as_numeric=True)
screening_table = screening_table.rename(
    columns={"R2": "CV R2", "MAE": "CV MAE", "RMSE": "CV RMSE"}
).drop(columns=["MSE", "MAPE"])
screening_table["fit_s"] = screening_table["Model"].map(screening_seconds).round(1)
screening_table = (
    screening_table[["Model", "CV R2", "CV MAE", "CV RMSE", "fit_s"]]
    .sort_values("CV RMSE")
    .reset_index(drop=True)
)

assert len(screening_table) == len(MODEL_NAMES), len(screening_table)
display(screening_table.round(4))

The table is ranked by **CV RMSE**, the selection criterion. Lower is better for both error columns; higher is
better for R². `fit_s` is the wall-clock cost of fitting and evaluating that candidate once, not a tuning budget.

A strong linear result would not be surprising: irradiation and output are close to proportional within a plant,
and the plant indicator absorbs the capacity difference. The question the table answers is whether any non-linear
model buys enough to justify its extra machinery.

## Chronological Holdout for Every Candidate

In [ ]:
holdout_table = pd.DataFrame(
    [
        {
            "Model": name,
            "Test R2": metrics["r2"]["test"],
            "Test MAE": metrics["mae"]["test"],
            "Test RMSE": metrics["rmse"]["test"],
        }
        for name, metrics in screening_metrics.items()
    ]
).sort_values("Test RMSE").reset_index(drop=True)

assert len(holdout_table) == len(MODEL_NAMES), len(holdout_table)
display(holdout_table.round(4))

## Reading the Two Tables

The holdout is one weather week. If its ranking differs from the CV ranking, that difference is the honest cost of
generalising to new days, not evidence that the holdout winner should be shipped. **Selection stays on CV RMSE**
because the holdout has to remain an uncontaminated report.

The two lowest-CV-RMSE candidates advance to tuning. Everything else is finished: its full record is already in
these two tables.

In [ ]:
TOP_TWO = screening_table.head(2)["Model"].tolist()
print("top two by CV RMSE:", TOP_TWO)
assert len(TOP_TWO) == 2

top_two_holdout = holdout_table[holdout_table.Model.isin(TOP_TWO)]
display(top_two_holdout.round(4))

## Detailed Diagnostics for the Finalists

The screening run deliberately produced no plots. The two finalists now get the full harness — 5-fold,
3-repeat cross-validation and the three diagnostic views:

- **Predicted vs actual** exposes systematic bias and compression at the top of the range.
- **Residuals vs predicted** shows whether error grows with output (heteroscedasticity) or bends (missed
  non-linearity).
- **Residual distribution** shows skew and heavy tails that a single RMSE hides.

In [ ]:
detailed_metrics = {}

for name in TOP_TWO:
    print(f"\n=== {name} ===")
    estimator = build_estimator(name).fit(X_train, y_train)
    detailed_metrics[name] = eval_regression(
        model=estimator,
        X_test=X_test,
        y_test=y_test,
        X_train=X_train,
        y_train=y_train,
        n_splits=5,
        n_repeats=3,
        display=True,
        show_plots=True,
    )

# **STAGE 5 : HYPERPARAMETER TUNING**

## Tuning Strategy

Only the two screening finalists are tuned. A randomized search is used rather than an exhaustive grid because
the trees' grids are large and most of the extra combinations buy noise, not skill.

- **30 sampled configurations per model**, scored by 5-fold, 3-repeat cross-validation on the training window.
- **Search and evaluation stay inside the training window.** The chronological holdout is touched only to report
  the final numbers.
- **Tuning is not allowed to make the artifact worse.** Each tuned model is compared with its own untuned baseline
  under the same 5-fold protocol; if tuning does not improve CV RMSE, the untuned pipeline ships and the bundle
  records that fact.

In [ ]:
from utils import get_hyperparameters, tune_pipelines

tuning_pipelines = {name: build_estimator(name) for name in TOP_TWO}
tuning_grids = {
    name: get_hyperparameters(MODEL_REGISTRY_KEYS[name], "random") for name in TOP_TWO
}

# `utils`' RandomForest grid still offers friedman_mse, which scikit-learn 1.9
# removed for RandomForestRegressor (it remains valid for GradientBoosting).
# Filtering here keeps the shared toolkit untouched.
if "RandomForest" in tuning_grids:
    tuning_grids["RandomForest"]["rf__criterion"] = ["squared_error", "absolute_error"]

tuned_searches, tuning_seconds = tune_pipelines(
    pipedict=tuning_pipelines,
    param_grid=tuning_grids,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    search_method="random",
    n_iter=30,
    n_splits=5,
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    display=False,
    plot_learning_curves=True,
    show=True,
)

print("fit seconds:", dict(zip(TOP_TWO, tuning_seconds)))
for name in TOP_TWO:
    print(f"{name} best params: {tuned_searches[name].best_params_}")

## Before and After Tuning

Both candidates are re-evaluated under the **same** 5-fold, 3-repeat protocol so the comparison is like-for-like.
`eval_regression` expects a fitted estimator, so each vanilla baseline is fitted explicitly before evaluation.

In [ ]:
def cv_metrics(name: str, estimator, n_splits: int = 5) -> dict:
    """Evaluate a fitted estimator with the confirmation protocol."""
    return eval_regression(
        model=estimator,
        X_test=X_test,
        y_test=y_test,
        X_train=X_train,
        y_train=y_train,
        n_splits=n_splits,
        n_repeats=3,
        plot_predicted_vs_actual=False,
        plot_residuals=False,
        plot_residuals_dist=False,
        display=False,
        show_plots=False,
    )


shipped_models = {}
shipping_rows = []

for name in TOP_TWO:
    vanilla = build_estimator(name).fit(X_train, y_train)
    vanilla_metrics = cv_metrics(name, vanilla)

    tuned = tuned_searches[name].best_estimator_
    tuned_metrics = cv_metrics(name, tuned)

    use_tuned = (
        tuned_metrics["cv"]["rmse"]["test_mean"]
        < vanilla_metrics["cv"]["rmse"]["test_mean"]
    )
    chosen = tuned if use_tuned else vanilla
    chosen_metrics = tuned_metrics if use_tuned else vanilla_metrics
    label = f"{name} (tuned)" if use_tuned else name

    shipped_models[name] = (chosen, chosen_metrics, label)
    shipping_rows.append(
        {
            "Model": name,
            "CV RMSE vanilla": vanilla_metrics["cv"]["rmse"]["test_mean"],
            "CV RMSE tuned": tuned_metrics["cv"]["rmse"]["test_mean"],
            "Holdout RMSE tuned": tuned_metrics["rmse"]["test"],
            "Shipped": "tuned" if use_tuned else "vanilla",
        }
    )

tuning_table = pd.DataFrame(shipping_rows).sort_values("CV RMSE tuned").reset_index(drop=True)

assert set(shipped_models) == set(TOP_TWO)
assert np.isfinite(tuning_table["CV RMSE tuned"]).all()
display(tuning_table.round(4))
print({name: label for name, (_, _, label) in shipped_models.items()})

**Reading the table:** a negative change from vanilla to tuned means tuning helped; a positive change would mean
the search found a worse configuration than the default, in which case the vanilla pipeline is shipped instead.
Either way, the artifact is the better of the two under the same cross-validation, and the bundle's `label`
records which one it is.

# **STAGE 6 : CONCLUSION & MODEL PERSISTENCE**

## Artifact Contract

Each shipped model is saved as a **bundle**, not a bare estimator. A bare pickle would carry no feature order, no
metrics, and no target name; a consumer could silently score the wrong columns and never know.

A bundle carries exactly what a downstream application needs:

| Key | Contents |
| --- | --- |
| `model` | The fitted estimator or pipeline (raw inputs, because any scaler is embedded). |
| `feature_columns` | The exact column order used at fit time. |
| `metrics` | `{"cv": {...}, "test": {...}}` — cross-validation and the chronological holdout. |
| `label` | Human-readable model name, including whether it was tuned. |
| `target` | `"AC_POWER"`. |

Next to the bundles, `manifest.json` lists which files are current. The application reads the manifest, so a later
run that changes the winner does not require editing application code.

In [ ]:
import json
import re
from datetime import datetime

import joblib

MODELS_DIR.mkdir(exist_ok=True)
MANIFEST_PATH = MODELS_DIR / "manifest.json"


def build_bundle(model, metrics: dict, label: str) -> dict:
    """Package a fitted model with everything a consumer needs to score a row."""
    return {
        "model": model,
        "feature_columns": list(FEATURE_COLUMNS),
        "metrics": {
            "cv": metrics["cv"],
            "test": {
                key: value["test"] for key, value in metrics.items() if key != "cv"
            },
        },
        "label": label,
        "target": TARGET,
    }


def slugify(label: str) -> str:
    """Turn a model label into a stable filename stem."""
    return re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")


bundle_files = {}
for name, (estimator, metrics, label) in shipped_models.items():
    path = MODELS_DIR / f"{slugify(label)}.joblib"
    joblib.dump(build_bundle(estimator, metrics, label), path)
    bundle_files[name] = path.name
    print(f"saved {path} | {len(FEATURE_COLUMNS)} features | {label}")

In [ ]:
manifest = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "target": TARGET,
    "feature_columns": list(FEATURE_COLUMNS),
    "split": {
        "train_end": str(train_df.DATE_TIME.max()),
        "test_start": str(test_df.DATE_TIME.min()),
    },
    "rows": {"train": int(len(train_df)), "test": int(len(test_df))},
    "models": [
        {"label": shipped_models[name][2], "file": bundle_files[name]}
        for name in shipped_models
    ],
}

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

assert MANIFEST_PATH.exists()
assert all((MODELS_DIR / entry["file"]).exists() for entry in manifest["models"])
assert len(manifest["models"]) == 2

for entry in manifest["models"]:
    bundle = joblib.load(MODELS_DIR / entry["file"])
    assert bundle["feature_columns"] == manifest["feature_columns"]
    assert bundle["target"] == TARGET

print(json.dumps(manifest, indent=2))

## Conclusion

**What shipped.** The screening finalists were GradientBoosting and RandomForest; both improved under tuning and
both were persisted. XGBoost was not assumed into the app: it ranked fourth on CV RMSE (R² ≈ 0.876) and did not
advance. The linear family finished far behind (CV R² ≈ 0.77), which is the clearest evidence that the
irradiation-to-power relationship is not linear once plant capacity and time of day are in play.

**How good is it.** On the chronological holdout the tuned trees explain roughly 89% of the variance in daylight
AC power, with RMSE near 1.95 MW on plants whose peak is 26-29 MW. That is a useful planning number and a modest
one: irradiation already carries most of the signal, so the model's contribution is the shape around it, not the
order of magnitude.

**What limits it.**

- **Thirty-four days, one season.** The holdout is a single weather week; a different week would move the number.
- **Nine Plant 2 outages** are unpredictable from weather and sit in the residuals by construction.
- **Plant 1's DC channel is unusable**, so no efficiency or DC-side analysis was attempted.
- **`PLANT_ID` is two-level.** The model interpolates between these two plants; it says nothing about a third.
- **Daylight only.** Night output is defined as zero, not predicted.
- **The split is chronological but single-window.** A rolling-origin evaluation over several weeks would give a
  distribution instead of one number.

**Next steps.** A longer history, forecast-error propagation (irradiance forecasts are not perfect inputs), and a
quantile model that reports a range for reserve scheduling rather than a point estimate.